# FT-Transformer

In [2]:
!python -m pip install --upgrade pip setuptools wheel

Defaulting to user installation because normal site-packages is not writeable


In [3]:
!pip install ipywidgets

Defaulting to user installation because normal site-packages is not writeable


In [4]:
!pip install rtdl_revisiting_models -q

In [5]:
!pip install torch torchvision torchaudio -i https://pypi.tuna.tsinghua.edu.cn/simple

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


In [6]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))

2.7.1+cu118
True
11.8
NVIDIA GeForce RTX 2070


In [7]:
import pandas as pd
import numpy as np
import os

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from rtdl_revisiting_models import FTTransformer

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

DATA_PATH = r"D:\墨大sml作业\FeatureA_Repeated"
OUTPUT_PATH = r"D:\墨大sml作业\Official_FTTransformer_FeatureA_Results"

os.makedirs(OUTPUT_PATH, exist_ok=True)

N_REPEATS = 10

cuda


In [9]:
def get_feature_cols(df):
    return [
        col for col in df.columns
        if col not in ["userId", "movieId", "label"]
    ]

In [10]:
class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [11]:
def compute_auc_from_scratch(y_true, y_score):
    y_true = np.array(y_true)
    y_score = np.array(y_score)

    sorted_indices = np.argsort(-y_score)
    y_true_sorted = y_true[sorted_indices]

    pos_count = np.sum(y_true == 1)
    neg_count = np.sum(y_true == 0)

    if pos_count == 0 or neg_count == 0:
        return 0

    tp = 0
    fp = 0

    tpr_list = [0]
    fpr_list = [0]

    for label in y_true_sorted:
        if label == 1:
            tp += 1
        else:
            fp += 1

        tpr_list.append(tp / pos_count)
        fpr_list.append(fp / neg_count)

    auc = 0

    for i in range(1, len(tpr_list)):
        auc += (
            (fpr_list[i] - fpr_list[i - 1])
            * (tpr_list[i] + tpr_list[i - 1])
            / 2
        )

    return auc


def compute_metrics_from_scratch(y_true, y_pred, y_score):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    auc = compute_auc_from_scratch(y_true, y_score)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn
    }

In [12]:
def stratified_sample_binary(df, sample_size, random_seed):
    pos_df = df[df["label"] == 1]
    neg_df = df[df["label"] == 0]

    pos_n = sample_size // 2
    neg_n = sample_size - pos_n

    pos_sample = pos_df.sample(
        n=pos_n,
        random_state=random_seed
    )

    neg_sample = neg_df.sample(
        n=neg_n,
        random_state=random_seed
    )

    sampled_df = pd.concat(
        [pos_sample, neg_sample],
        axis=0
    ).sample(
        frac=1,
        random_state=random_seed
    ).reset_index(drop=True)

    return sampled_df

In [13]:
def stratified_split_from_scratch(df, label_col, test_ratio=0.2, random_seed=42):
    rng = np.random.default_rng(random_seed)

    train_indices = []
    test_indices = []

    for label_value in df[label_col].unique():
        label_indices = df[df[label_col] == label_value].index.to_numpy()
        rng.shuffle(label_indices)

        test_size = int(len(label_indices) * test_ratio)

        test_indices.extend(label_indices[:test_size])
        train_indices.extend(label_indices[test_size:])

    train_df = df.loc[train_indices].sample(
        frac=1,
        random_state=random_seed
    ).reset_index(drop=True)

    test_df = df.loc[test_indices].sample(
        frac=1,
        random_state=random_seed
    ).reset_index(drop=True)

    return train_df, test_df

In [14]:
def train_official_ft_transformer(
    train_df,
    test_df,
    lr=1e-4,
    weight_decay=1e-5,
    batch_size=4096,
    epochs=3
):
    feature_cols = get_feature_cols(train_df)

    X_train = train_df[feature_cols].values
    y_train = train_df["label"].values

    X_test = test_df[feature_cols].values
    y_test = test_df["label"].values

    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    train_dataset = TabularDataset(X_train, y_train)
    test_dataset = TabularDataset(X_test, y_test)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    model = FTTransformer(
        n_cont_features=len(feature_cols),
        cat_cardinalities=[],
        d_out=1,
        **FTTransformer.get_default_kwargs()
    ).to(device)

    optimizer = model.make_default_optimizer()
    
    # Override default optimizer lr / weight_decay if needed
    for group in optimizer.param_groups:
        group["lr"] = lr
        group["weight_decay"] = weight_decay

    criterion = nn.BCEWithLogitsLoss()

    model.train()

    for epoch in range(epochs):
        total_loss = 0

        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()

            logits = model(batch_X, None).squeeze(1)

            loss = criterion(logits, batch_y)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch + 1} loss:", round(total_loss, 4))

    model.eval()

    all_preds = []
    all_scores = []
    all_labels = []

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X = batch_X.to(device)

            logits = model(batch_X, None).squeeze(1)
            probs = torch.sigmoid(logits)

            preds = (probs >= 0.5).int()

            all_preds.extend(preds.detach().cpu().tolist())
            all_scores.extend(probs.detach().cpu().tolist())
            all_labels.extend(batch_y.detach().cpu().tolist())

    metrics = compute_metrics_from_scratch(
        all_labels,
        all_preds,
        all_scores
    )

    return metrics

In [15]:
TRAIN_SAMPLE_SIZE = 30000
TEST_SAMPLE_SIZE = 10000

LR_VALUES = [5e-5, 1e-4, 5e-4]
WEIGHT_DECAY_VALUES = [1e-5]

N_INNER_REPEATS = 3
VALID_RATIO = 0.2

BATCH_SIZE = 1024
EPOCHS = 3


all_results = []
all_tuning_results = []

for repeat_id in range(1, N_REPEATS + 1):
    print("=" * 60)
    print(f"Outer Repeat {repeat_id:02d}")
    print("=" * 60)

    repeat_folder = os.path.join(
        DATA_PATH,
        f"repeat_{repeat_id:02d}"
    )

    train_path = os.path.join(repeat_folder, "feature_A_train.csv")
    test_path = os.path.join(repeat_folder, "feature_A_test.csv")

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    train_df = stratified_sample_binary(
        train_df,
        sample_size=TRAIN_SAMPLE_SIZE,
        random_seed=42 + repeat_id
    )

    test_df = stratified_sample_binary(
        test_df,
        sample_size=TEST_SAMPLE_SIZE,
        random_seed=100 + repeat_id
    )

    print("Outer train shape:", train_df.shape)
    print("Outer test shape:", test_df.shape)

    best_lr = None
    best_weight_decay = None
    best_mean_val_f1 = -1

    for lr in LR_VALUES:
        for weight_decay in WEIGHT_DECAY_VALUES:

            inner_f1_scores = []

            for inner_id in range(N_INNER_REPEATS):
                inner_train_df, val_df = stratified_split_from_scratch(
                    train_df,
                    label_col="label",
                    test_ratio=VALID_RATIO,
                    random_seed=2000 + repeat_id * 10 + inner_id
                )

                print(
                    f"Trying lr={lr}, weight_decay={weight_decay}, inner={inner_id + 1}"
                )

                val_metrics = train_official_ft_transformer(
                    train_df=inner_train_df,
                    test_df=val_df,
                    lr=lr,
                    weight_decay=weight_decay,
                    batch_size=BATCH_SIZE,
                    epochs=EPOCHS
                )

                inner_f1_scores.append(val_metrics["f1"])

            mean_val_f1 = np.mean(inner_f1_scores)
            std_val_f1 = np.std(inner_f1_scores, ddof=1)

            all_tuning_results.append({
                "outer_repeat": repeat_id,
                "lr": lr,
                "weight_decay": weight_decay,
                "mean_validation_f1": mean_val_f1,
                "std_validation_f1": std_val_f1
            })

            print("Mean validation F1:", round(mean_val_f1, 6))

            if mean_val_f1 > best_mean_val_f1:
                best_mean_val_f1 = mean_val_f1
                best_lr = lr
                best_weight_decay = weight_decay

    print("Best lr:", best_lr)
    print("Best weight_decay:", best_weight_decay)
    print("Best mean validation F1:", best_mean_val_f1)

    test_metrics = train_official_ft_transformer(
        train_df=train_df,
        test_df=test_df,
        lr=best_lr,
        weight_decay=best_weight_decay,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS
    )

    final_result = {
        "repeat": repeat_id,
        "best_lr": best_lr,
        "best_weight_decay": best_weight_decay,
        "batch_size": BATCH_SIZE,
        "best_mean_val_f1": best_mean_val_f1,
        **test_metrics
    }

    all_results.append(final_result)

    print("Accuracy :", round(test_metrics["accuracy"], 4))
    print("Precision:", round(test_metrics["precision"], 4))
    print("Recall   :", round(test_metrics["recall"], 4))
    print("F1       :", round(test_metrics["f1"], 4))
    print("AUC      :", round(test_metrics["auc"], 4))

Outer Repeat 01
Outer train shape: (30000, 17)
Outer test shape: (10000, 17)
Trying lr=5e-05, weight_decay=1e-05, inner=1
Epoch 1 loss: 14.3566
Epoch 2 loss: 13.4745
Epoch 3 loss: 13.373
Trying lr=5e-05, weight_decay=1e-05, inner=2
Epoch 1 loss: 14.4343
Epoch 2 loss: 13.5478
Epoch 3 loss: 13.3955
Trying lr=5e-05, weight_decay=1e-05, inner=3
Epoch 1 loss: 14.4519
Epoch 2 loss: 13.5254
Epoch 3 loss: 13.3705
Mean validation F1: 0.736712
Trying lr=0.0001, weight_decay=1e-05, inner=1
Epoch 1 loss: 14.4977
Epoch 2 loss: 13.4593
Epoch 3 loss: 13.2938
Trying lr=0.0001, weight_decay=1e-05, inner=2
Epoch 1 loss: 14.301
Epoch 2 loss: 13.4357
Epoch 3 loss: 13.3061
Trying lr=0.0001, weight_decay=1e-05, inner=3
Epoch 1 loss: 14.1171
Epoch 2 loss: 13.361
Epoch 3 loss: 13.2424
Mean validation F1: 0.730943
Trying lr=0.0005, weight_decay=1e-05, inner=1
Epoch 1 loss: 14.7124
Epoch 2 loss: 13.3173
Epoch 3 loss: 13.2686
Trying lr=0.0005, weight_decay=1e-05, inner=2
Epoch 1 loss: 14.4832
Epoch 2 loss: 13.39

In [16]:
results_df = pd.DataFrame(all_results)
tuning_results_df = pd.DataFrame(all_tuning_results)

results_path = os.path.join(
    OUTPUT_PATH,
    "FTTransformer_FeatureA_repeated_results.csv"
)

tuning_path = os.path.join(
    OUTPUT_PATH,
    "FTTransformer_FeatureA_tuning_results.csv"
)

results_df.to_csv(results_path, index=False, encoding="utf-8-sig")
tuning_results_df.to_csv(tuning_path, index=False, encoding="utf-8-sig")

print("Saved repeated test results to:")
print(results_path)

print("Saved tuning results to:")
print(tuning_path)

results_df

Saved repeated test results to:
D:\墨大sml作业\Official_FTTransformer_FeatureA_Results\FTTransformer_FeatureA_repeated_results.csv
Saved tuning results to:
D:\墨大sml作业\Official_FTTransformer_FeatureA_Results\FTTransformer_FeatureA_tuning_results.csv


,repeat,best_lr,best_weight_decay,batch_size,best_mean_val_f1,accuracy,precision,recall,f1,auc,tp,tn,fp,fn
0,1,0.00005,0.00001,1024,0.736712,0.7131,0.701914,0.7408,0.720833,0.790741,3704,3427,1573,1296
1,2,0.00005,0.00001,1024,0.733065,0.7180,0.698688,0.7666,0.731070,0.787873,3833,3347,1653,1167
2,3,0.00010,0.00001,1024,0.739156,0.7201,0.719705,0.7210,0.720352,0.796064,3605,3596,1404,1395
3,4,0.00050,0.00001,1024,0.734402,0.7109,0.683232,0.7864,0.731195,0.787358,3932,3177,1823,1068
4,5,0.00005,0.00001,1024,0.731138,0.7139,0.691016,0.7738,0.730069,0.788332,3869,3270,1730,1131
5,6,0.00010,0.00001,1024,0.735696,0.7168,0.688587,0.7916,0.736509,0.792714,3958,3210,1790,1042
6,7,0.00010,0.00001,1024,0.724168,0.7057,0.728201,0.6564,0.690439,0.782360,3282,3775,1225,1718
7,8,0.00010,0.00001,1024,0.724706,0.7234,0.716221,0.7400,0.727917,0.795864,3700,3534,1466,1300
8,9,0.00005,0.00001,1024,0.726612,0.7164,0.707518,0.7378,0.722342,0.789458,3689,3475,1525,1311
9,10,0.00005,0.00001,1024,0.736318,0.7141,0.702248,0.7434,0.722238,0.783362,3717,3424,1576,1283


In [17]:
summary_records = []

for metric in ["accuracy", "precision", "recall", "f1", "auc"]:
    values = results_df[metric].values

    summary_records.append({
        "metric": metric,
        "mean": np.mean(values),
        "std": np.std(values, ddof=1),
        "standard_error": np.std(values, ddof=1) / np.sqrt(len(values))
    })

summary_df = pd.DataFrame(summary_records)

summary_path = os.path.join(
    OUTPUT_PATH,
    "FTTransformer_FeatureA_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig"
)

summary_df

,metric,mean,std,standard_error
0,accuracy,0.715240,0.004934,0.001560
1,precision,0.703733,0.014401,0.004554
2,recall,0.745780,0.039013,0.012337
3,f1,0.723296,0.012741,0.004029
4,auc,0.789413,0.004627,0.001463


In [18]:
# Best learning rate frequency for FT-Transformer

best_lr_frequency = (
    results_df["best_lr"]
    .value_counts()
    .reset_index()
)

best_lr_frequency.columns = ["learning_rate", "frequency"]

best_lr_frequency_path = os.path.join(
    OUTPUT_PATH,
    "FTTransformer_FeatureA_best_lr_frequency.csv"
)

best_lr_frequency.to_csv(
    best_lr_frequency_path,
    index=False,
    encoding="utf-8-sig"
)


best_lr_frequency

,learning_rate,frequency
0,0.00005,5
1,0.00010,4
2,0.00050,1
